In [1]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

Function to read File and Downsample

In [3]:
def read_and_downsample(path,file):
    fullpath=path+str(file)+'.csv'
    csv = pd.read_csv(fullpath, skip_blank_lines=True,usecols=['Load (N)','Displacement (mm)'])
    csv=csv[['Displacement (mm)','Load (N)']]
    csv=csv.dropna(how="all")
    ida=csv['Load (N)'].idxmax()
    max_disp=csv['Displacement (mm)'][ida]
    points=round(max_disp/10)+1
    Xresampled = np.linspace(0,max_disp,points)
    csv.set_index('Displacement (mm)',inplace =True)
    csv=csv[~csv.index.duplicated(keep='first')]
    csv_resampled = csv.reindex(csv.index.union(Xresampled)).interpolate('values').loc[Xresampled]
    return csv, csv_resampled, points

Function to plot and compare two dataframes

In [5]:
def compare_plots(df1,df2,tags,axis):
    x=axis[0]
    y=axis[1]
    sns.set_theme()
    concatenated = pd.concat([df1.assign(dataset=tags[0]), df2.assign(dataset=tags[1])])

    fig, ax = plt.subplots()
    sns.lineplot(data=concatenated, x=x, y=y,style='dataset',
                 markers={tags[0]: ',', tags[1]: 'o'},hue='dataset')

Let's interpret results for Sample 22 and compare the results before and after downsampling

## For sample 22

tensile_path="C://Users//nnuno//Desktop//PhD//Fantomas//Skin//Ensaios//SiliconeC//"
s22,s22_res,points=read_and_downsample(tensile_path,22)
compare_plots(s22,s22_res,['Original','Downsampled'],['Displacement (mm)','Load (N)'])

Now that we know the exact number of points, let's process the images for this sample, so we can find out the width change

In [10]:
import cv2
import matplotlib.pyplot as plt
import os
import pandas as pd
import numpy as np

Let's admit all images have the same ppm, since camera was always in same position

Function to do the first crop

In [13]:
def crop1(image,array):
    #array=[t,b,l,r]
    height, width = image.shape
    # Setting the points for cropped image
    top = int(array[0]*height)
    bottom =int(array[1]*height)
    left = int(array[2]*width)
    right = int(array[3]*width)
    return image[top:bottom, left:right]

Function to do the sample crop

In [19]:
def crop2(image,n):
    if image is None or image.size == 0:
            raise ValueError("Invalid input image provided (Crop1)")
    wlc = cv2.reduce(image, 1, cv2.REDUCE_SUM, dtype=cv2.CV_32SC1)/255 #white line count
    blc=[len(image[0])-i[0] for i in wlc] #black line count
    limits=[]
    i=0
    while blc[i]>n:
        i+=1
    limits.append(i/len(image))
    while blc[i]<n:
        i+=1
    limits.append(i/len(image))
    limits.append(3/8)
    limits.append(5/8)
    return crop1(image,limits)

Function to do return number of of black pixels in a line as an array of lines

In [31]:
def find_blc(image):
    if image is None or image.size == 0:
        raise ValueError("Invalid input image provided")
    wlc = cv2.reduce(image, 1, cv2.REDUCE_SUM, dtype=cv2.CV_32SC1)
    #wlc[wlc=='None']=0
    #if wlc is None: wlc=0
    wlc=wlc/255
    return [len(image[0])-i for i in wlc]

Find the smaller number of black pixels measured

In [19]:
def measure(image,ppm):
    return (min(find_blc(image))/ppm)

Make the graph compare the images form each frame

In [21]:
def compare_image_global(i,a1,a2,a3,a4,txt):
    text='Frame '+str(i)
    image=[a1,a2,a3,a4]
    aspect=[]
    for img in image:
        aspect.append(img.shape[1] / img.shape[0])
    texts=['Original',
          'First Crop',
          'BW',
          'Sample Crop']
    f, axarr = plt.subplots(1,4, figsize=(14, 4))
    for j in range(len(image)):
        axarr[j].imshow(image[j], cmap='gray', vmin=0, vmax=255,aspect=aspect[j])
        axarr[j].set_title(texts[j])
        axarr[j].axis('off')
    f.suptitle(text, fontsize=30)
    plt.figtext(0.5, 0, txt, wrap=True, horizontalalignment='center', fontsize=10)

Make a list of the resampled data frames

In [23]:
def resample_frames(points,path):
    df=pd.DataFrame(os.listdir(path))
    return np.linspace(0,len(df)-1,points, dtype=int)

Function that will process all frames and let us visualize the data

In [44]:
def process_camera(frames,path,vel,fps,ppm,thresh,n):
    length=[]
    width=[]
    for i in range(len(frames)):
        #read image
        path_image=path+'PST_3-sys1-'+"{:04d}".format(frames[i])+'_0.tif'
        print(path_image)
        im_gray = cv2.imread(path_image, cv2.IMREAD_GRAYSCALE)
        if im_gray is None or im_gray.size == 0:
            raise ValueError("Invalid input image provided (gray)")
        #increase upper limit
        top_extra=(vel*(1/60)*(1/fps)*frames[i]*ppm)/len(im_gray)
        #first crop
        limits=[8/15-top_extra,11/15,1/3,2/3]
        im1 = crop1(im_gray,limits)
        if im1 is None or im1.size == 0:
            raise ValueError("Invalid input image provided (Crop1)")
        #apply BW
        im_bw = cv2.threshold(im1, thresh, 255, cv2.THRESH_BINARY)[1]
        #second crop
        im2=crop2(im_bw,n)
        if im2 is None or im2.size == 0:
            raise ValueError("Invalid input image provided (Crop2)")
        #visualize data
        length.append(len(im2)/ppm)
        cur_width=min(find_blc(im2))/ppm
        if i>1:
            if cur_width>width[i-1]:width.append(width[i-1])
            else:width.append(cur_width)
        else:width.append(cur_width)
        txt='Length='+str(length[i])+' mm \nWidth='+str(width[i])+' mm \n'
        compare_image_global(frames[i],im_gray,im1,im_bw,im2,txt)
    return length, width

ppm=3.4106412
velocity=10 #mm/min
fps=0.5 #1 frame every 2000 ms
camera_path='C://Users//nnuno//Desktop//PhD//Fantomas//Skin//Camara//22//Process//'
frames=resample_frames(points,camera_path)
print(frames)
thresh=30
blc_limit=100
l,w=process_camera(frames,camera_path,velocity,fps,ppm,thresh,blc_limit)

In [27]:
def calculate_ss(resample,length,width,init_thick):
    init_length=length[0]
    init_width=width[0]
    area=init_width*init_thick
    estim_area=init_width*init_thick
    stress_strain=pd.DataFrame()
    eng_stress_strain=pd.DataFrame()
    strain=[]
    stress=[]
    eng_stress=[]
    latstrain=[]
    for i in range(resample.shape[0]):
        cur_load=resample.iloc[i]['Load (N)']
        cur_long_strain=resample.index[resample['Load (N)'] == cur_load].tolist()[0]/init_length
        cur_lat_strain=(init_width-width[i])/init_width
        new_area=((1-cur_lat_strain)**2)*init_width*init_thick
        strain.append(cur_long_strain)
        stress.append(float(cur_load/new_area))
        eng_stress.append(float(cur_load/area))
    stress_strain['Strain'] = strain
    stress_strain['Stress (MPa)']= stress
    eng_stress_strain['Strain'] = strain
    eng_stress_strain['Stress (MPa)']= eng_stress
    return stress_strain,eng_stress_strain

i_thick=2.12 #mm
ss,eng_ss=calculate_ss(s22_res,l,w,i_thick)

compare_plots(eng_ss,ss,['Engineering Stress','True Stress'],['Strain','Stress (MPa)'])

Now let's do the same for all samples!